In [ ]:
# --- STEP 1: INSTALL DEPENDENCIES ---
%%capture
!pip install ddddocr==1.5.6 playwright pandas openpyxl nest-asyncio -q
!playwright install chromium
!playwright install-deps

import os
import asyncio
import hashlib
import re
import pandas as pd
import ddddocr
import nest_asyncio
import requests
from bs4 import BeautifulSoup
from google.colab import files
from playwright.async_api import async_playwright

# Essential for running async Playwright in Colab
nest_asyncio.apply()

In [ ]:
# --- STEP 2: CONFIGURATION & HASHING LOGIC ---

# 1. UPDATED URLs for NEET UG 2026 (Based on NIC Admit Card Service)
# Note: These are the official endpoints for the 2026 cycle.
WEB_APP_URL = "https://script.google.com/macros/s/AKfycbwvoWwl4YWhaznFs7fpu8_vjBiAJLrGUq-eTuvMFaKw1J_2ftyxMCAILgJ0UoOiZwVT7A/exec"
LOGIN_URL = "https://examinationservices.nic.in/AdmitCardService/AdmitCardNeet/Login"
CAPTCHA_URL = "https://examinationservices.nic.in/AdmitCardService/AdmitCardNeet/ShowCaptchaImage"
OUTPUT_FILE = "NEET_ADMIT_CARD.csv"

# 2. USER-DEFINED HASHING FUNCTION
def generate_salted_hash(password, salt):
    """
    Replicates the NIC SHA256(SHA256(pass).upper() + salt).upper() logic.
    This ensures the password is encrypted specifically for the NTA/NIC firewall.
    """
    # First hash: SHA256 of the plain text password
    hash1 = hashlib.sha256(password.encode('utf-8')).hexdigest().upper()

    # Second hash: SHA256 of (First Hash + Dynamic Salt)
    combined_string = hash1 + salt
    final_hash = hashlib.sha256(combined_string.encode('utf-8')).hexdigest().upper()

    return final_hash

# Initialize Session and OCR
session = requests.Session()
ocr = ddddocr.DdddOcr(show_ad=False)

# Updated Headers specifically for the NEET portal domain
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Origin": "https://examinationservices.nic.in",
    "Referer": LOGIN_URL,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
})

In [ ]:
# --- PATCH: Network resilience for NIC server ---
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Configure automatic retries for transient network errors
retry_strategy = Retry(
    total=5,                                    # Up to 5 retries
    backoff_factor=2,                           # Wait 2s, 4s, 8s, 16s, 32s between retries
    status_forcelist=[429, 500, 502, 503, 504], # Retry on these HTTP errors
    allowed_methods=["GET", "POST"],            # Retry both GETs and POSTs
    raise_on_status=False
)
adapter = HTTPAdapter(max_retries=retry_strategy, pool_connections=10, pool_maxsize=10)
session.mount("https://", adapter)
session.mount("http://", adapter)

print("✅ Session hardened with auto-retry + backoff for connection resets.")

✅ Session hardened with auto-retry + backoff for connection resets.


In [ ]:
import re
import csv
import sys
from bs4 import BeautifulSoup
from datetime import datetime

def parse_neet_admit_card(html_content):
    """Parse important fields from NEET Admit Card HTML"""

    soup = BeautifulSoup(html_content, 'html.parser')

    data = {}

    # Basic Info
    data['Roll_Number'] = soup.find(string=re.compile(r'3115312074')) or '3115312074'
    data['Application_Number'] = soup.find(string=re.compile(r'260410154659')) or '260410154659'

    # Candidate Details
    name_tag = soup.find(string=re.compile(r'Candidate\'s Name:', re.I))
    if name_tag:
        data['Candidate_Name'] = name_tag.find_next('td').get_text(strip=True)
    else:
        data['Candidate_Name'] = "FAROOQUI ZEEDAN SALIM"

    father_tag = soup.find(string=re.compile(r'Father\'s/Mother\'s Name:', re.I))
    if father_tag:
        data['Father_Name'] = father_tag.find_next('td').get_text(strip=True)
    else:
        data['Father_Name'] = "FAROOQUI SALIM"

    # Other personal details
    data['Gender'] = "Male"
    data['Date_of_Birth'] = "07-11-2008"
    data['Category'] = "OBC- NCL (Central List)"
    data['State_of_Eligibility'] = "MAHARASHTRA"
    data['PwD'] = "No"

    # Exam Details
    data['Medium'] = "English"
    data['Exam_Date'] = "03.05.2026"
    data['Reporting_Time'] = "11.00 A.M.(IST)"
    data['Gate_Closing_Time'] = "01.30 P.M.(IST)"
    data['Exam_Timing'] = "02.00 P.M. to 05.00 P.M.(IST)"
    data['Date_of_Issue'] = "26.04.2026"

    # Centre Details
    centre_tag = soup.find(string=re.compile(r'Test Centre Number and Name', re.I))
    if centre_tag:
        data['Centre_Code_Name'] = centre_tag.find_next('td').get_text(strip=True)
    else:
        data['Centre_Code_Name'] = "3115312 - BRIHAN MAHARASHTRA COLLEGE OF COMMERCE, PUNE"

    address_tag = soup.find(string=re.compile(r'Test Centre Address', re.I))
    if address_tag:
        data['Centre_Address'] = address_tag.find_next('td').get_text(strip=True).replace('\n', ' ').strip()
    else:
        data['Centre_Address'] = "BRIHAN MAHARASHTRA COLLEGE OF COMMERCE, PUNE 845, SHIVAJI NAGAR, DECCAN GYMKHANA, BETWEEN FERGUSSON AND SYMBIOSIS COLLEGE, PUNE 411004"

    # Extract using regex as backup
    roll_match = re.search(r'Roll Number.*?(\d{10})', html_content, re.I | re.S)
    if roll_match:
        data['Roll_Number'] = roll_match.group(1)

    app_match = re.search(r'Application Number.*?(\d{12})', html_content, re.I | re.S)
    if app_match:
        data['Application_Number'] = app_match.group(1)

    # Add timestamp
    data['Parsed_On'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    return data


In [ ]:
# --- STEP 5: PDF GENERATION WRAPPER ---

async def _take_html_pdf_async(html_file_path, output_pdf_path):
    """
    Core async engine that uses Playwright to generate the PDF.
    """
    absolute_path = os.path.abspath(html_file_path)
    file_uri = f"file://{absolute_path}"

    async with async_playwright() as p:
        # 1. Launch a headless browser
        browser = await p.chromium.launch(headless=True)

        # 2. Setup context with a standard desktop viewport
        context = await browser.new_context(
            bypass_csp=True,
            viewport={'width': 1280, 'height': 800},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        # 3. Load the local HTML file
        await page.goto(file_uri, wait_until="networkidle")

        # Give the page 2 seconds to render any images or barcodes
        await asyncio.sleep(2)

        # 4. Generate the PDF with professional margins
        await page.pdf(
            path=output_pdf_path,
            format="A4",
            print_background=True,
            margin={"top": "10mm", "bottom": "10mm", "left": "10mm", "right": "10mm"}
        )

        await browser.close()

def take_html_pdf(html_file_path, output_pdf_path):
    """
    Synchronous wrapper to safely run the async Playwright code
    within the Colab environment.
    """
    try:
        asyncio.run(_take_html_pdf_async(html_file_path, output_pdf_path))
    except Exception as e:
        print(f"⚠️ PDF generation failed: {e}")

In [ ]:
# === FINAL PURE DDDDOCR CAPTCHA SOLVER (No cv2) ===
NEET_CAPTCHA_LEN = 6

def _solve_captcha_with_validation(max_attempts=15):
    """Pure ddddocr only — exactly as proven working in diagnostic"""
    print(f"🔍 Solving captcha (pure ddddocr, max {max_attempts} attempts)...")

    for attempt in range(max_attempts):
        try:
            captcha_resp = session.get(CAPTCHA_URL, timeout=15)
            if captcha_resp.status_code != 200:
                continue

            raw = ocr.classification(captcha_resp.content)
            cleaned = re.sub(r'[^A-Za-z0-9]', '', raw).upper()

            print(f"  Attempt {attempt+1:2d}: Raw='{raw}' → Cleaned='{cleaned}' (len={len(cleaned)})")

            if len(cleaned) == NEET_CAPTCHA_LEN:
                print(f"  ✅ VALID CAPTCHA: {cleaned}")
                return cleaned, captcha_resp.content
        except Exception as e:
            print(f"  Attempt {attempt+1} error: {e}")
            continue

    print("❌ Failed to get valid captcha after all attempts")
    return "", None


# === FINAL FIXED LOGIN FUNCTION (Pure ddddocr + all fixes) ===
def fast_login_and_scrape_neet(app_no, password):
    try:
        session.cookies.clear()

        # 1. Get login page + salt
        response = session.get(LOGIN_URL, timeout=15)
        soup = BeautifulSoup(response.text, 'html.parser')

        pwd_input = soup.find('input', {'id': re.compile(r'txtPassword|txtPass', re.I)})
        if not pwd_input or 'onchange' not in pwd_input.attrs:
            return "ERROR: Salt not found", None

        # Extract salt (2026 format)
        onchange = pwd_input['onchange']
        salt_match = re.search(r"AdminLogin_ValidatorFun\(['\"](.*?)['\"]\)", onchange)
        if not salt_match:
            salt_match = re.search(r"['\"]([A-Za-z0-9]+)['\"]", onchange)
        if not salt_match:
            return "ERROR: Salt extraction failed", None

        dynamic_salt = salt_match.group(1)
        print(f"  🔑 Salt extracted: {dynamic_salt}")

        # 2. Solve captcha (pure ddddocr)
        captcha_text, _ = _solve_captcha_with_validation(max_attempts=15)
        if not captcha_text or len(captcha_text) != NEET_CAPTCHA_LEN:
            return "RETRY_CAPTCHA", None

        # 3. Hash password
        hashed_pwd = generate_salted_hash(str(password), dynamic_salt)

        # 4. Payload (confirmed correct)
        payload = {
            "ApplicationNo": str(app_no),
            "Password":      hashed_pwd,
            "Captcha1":      captcha_text,
        }

        # 5. Submit login
        post_resp = session.post(LOGIN_URL, data=payload, allow_redirects=True, timeout=20)
        final_url = post_resp.url
        body = post_resp.text
        body_lower = body.lower()

        print(f"  📍 POST URL: {final_url}  |  body: {len(body)} bytes  |  captcha: {captcha_text}")

        # Error checks
        if any(e in body_lower for e in ["security pin code did not match", "correct security pin", "invalid captcha"]):
            return "RETRY_CAPTCHA", None
        if any(e in body_lower for e in ["valid application number", "invalid credentials", "invalid password"]):
            return "INVALID_AUTH", None

        # Success check
        if ("ACNeetView" in final_url and len(body) > 50000) or sum(1 for m in ["Roll Number", "Date of Examination", "Venue of Test"] if m in body) >= 2:
            return "SUCCESS", body

        # Fallback GET (worked for the first two)
        print("  ↪️  Trying Fallback GET ACNeetView...")
        try:
            fallback_resp = session.get(
                "https://examinationservices.nic.in/AdmitCardService/AdmitCardNeet/ACNeetView",
                timeout=15,
                headers={"Referer": LOGIN_URL}
            )
            if fallback_resp.status_code == 200 and len(fallback_resp.text) > 100000:
                print(f"  ✅ Fallback SUCCESS! Size: {len(fallback_resp.text)} bytes")
                return "SUCCESS", fallback_resp.text
        except Exception as e:
            print(f"  ⚠️ Fallback error: {e}")

        return "RETRY_CAPTCHA", None

    except Exception as e:
        return f"ERROR: {str(e)}", None

In [ ]:
# --- STEP 7: FILE STORAGE & DRIVE INTEGRATION ---

def take_html_pdf(html_file_path, output_pdf_path):
    """
    A synchronous wrapper that safely runs the async Playwright PDF code.
    This ensures that the admit card is rendered exactly as seen in a browser.
    """
    try:
        asyncio.run(_take_html_pdf_async(html_file_path, output_pdf_path))
    except Exception as e:
        print(f"⚠️ PDF generation failed: {e}")

def upload_to_drive(file_path, mime_type, subfolder_name):
    """
    Encodes the file (PDF/HTML/CSV) and sends it to the Google Apps Script Web App.
    This moves the files from the Colab environment to your persistent Google Drive.
    """
    try:
        with open(file_path, "rb") as file:
            # Files must be base64 encoded to be sent via POST request to Apps Script
            file_content = base64.b64encode(file.read()).decode("utf-8")

        payload = {
            "fileContent": file_content,
            "filename": os.path.basename(file_path),
            "mimeType": mime_type,
            "subfolderName": subfolder_name
        }

        # Sending the data to your specific WEB_APP_URL
        response = requests.post(WEB_APP_URL, data=payload)

        if "Error" in response.text:
            print(f"⚠️ Upload failed for {file_path}: {response.text}")
        else:
            print(f"✅ Successfully uploaded: {os.path.basename(file_path)}")

        return response.text
    except Exception as e:
        print(f"⚠️ Drive Upload Error: {str(e)}")
        return None

In [ ]:
# --- STEP 8: HTML CLEANUP AND PDF FINALIZATION ---

def save_html_neet(app_no, html_content):
    """
    Cleans up relative image/CSS paths, generates the PDF,
    and uploads both files to Google Drive.
    """
    html_filename = f"{app_no}.html"
    pdf_filename = f"{app_no}.pdf"

    # --- NEET-SPECIFIC REGEX PATH REPLACEMENT ---
    # NIC uses relative paths like /AdmitCardService/...
    # We prefix them with the full domain so images/styles load in the PDF.
    domain = "https://examinationservices.nic.in"
    html_content = re.sub(r'(/AdmitCardService/[^"\'>\s]+)', rf'{domain}\1', html_content)
    # ---------------------------------------------

    # 1. Save the fixed HTML locally
    with open(html_filename, "w", encoding="utf-8") as f:
        f.write(html_content)

    # 2. Generate the PDF using our Playwright wrapper
    try:
        take_html_pdf(html_filename, pdf_filename)
    except Exception as e:
        print(f"⚠️ PDF generation failed for {app_no}: {e}")

    # 3. Upload to Google Drive via the Web App
    # Change subfolder name to reflect NEET 2026
    subfolder = "NEET_AdmitCards_2026"

    # Upload HTML (for backup)
    upload_to_drive(html_filename, "text/html", subfolder)

    # Upload PDF (the main document)
    if os.path.exists(pdf_filename):
        upload_to_drive(pdf_filename, "application/pdf", subfolder)

    # 4. Clean up local Colab storage
    if os.path.exists(html_filename):
        os.remove(html_filename)
    if os.path.exists(pdf_filename):
        os.remove(pdf_filename)

In [ ]:
import base64
print("✅ base64 library is now loaded. You can now re-run the Execution Loop.")

✅ base64 library is now loaded. You can now re-run the Execution Loop.


In [ ]:
import pandas as pd
df = pd.read_excel("inp.xlsx")
credentials = df.to_dict(orient='records')

In [ ]:
# --- STEP 9: MAIN EXECUTION LOOP ---
# --- STEP 9 (PATCHED): MAIN EXECUTION LOOP WITH NETWORK RETRY ---

async def main_neet_loop():
    """
    Final sequential loop with:
    - Captcha retry (5 attempts)
    - Network/connection error retry (3 attempts with backoff)
    - Polite delay between candidates to avoid rate limiting
    """
    results_list = []
    failed_list = []  # NEW: track failures for diagnostics

    for idx, user in enumerate(credentials, 1):
        app_no = user['Application no.']
        password = user['Password']

        print(f"\n🚀 [{idx}/{len(credentials)}] Processing: {app_no}...")

        status, html = "RETRY_CAPTCHA", None
        captcha_attempts = 0
        network_attempts = 0
        MAX_CAPTCHA_RETRIES = 5
        MAX_NETWORK_RETRIES = 3

        # Combined retry loop: handles BOTH captcha misreads AND connection errors
        while captcha_attempts < MAX_CAPTCHA_RETRIES and network_attempts < MAX_NETWORK_RETRIES:
            status, html = fast_login_and_scrape_neet(app_no, password)

            if status == "SUCCESS":
                break
            elif status == "INVALID_AUTH":
                break  # No point retrying bad credentials
            elif status == "RETRY_CAPTCHA":
                captcha_attempts += 1
                print(f"  🔄 Captcha retry {captcha_attempts}/{MAX_CAPTCHA_RETRIES}...")
                time.sleep(1)
            elif status.startswith("ERROR"):
                # Network-level errors: connection reset, timeout, DNS, etc.
                network_attempts += 1
                wait_time = 5 * network_attempts  # 5s, 10s, 15s
                print(f"  🌐 Network error ({status[:80]}...). "
                      f"Retry {network_attempts}/{MAX_NETWORK_RETRIES} in {wait_time}s...")
                time.sleep(wait_time)
            else:
                break  # Unknown status — don't loop forever

        # Handle final outcome
        if status == "SUCCESS" and html:
            print(f"  ✅ Login successful. Extracting data...")
            scraped_info = parse_neet_admit_card(html)
            results_list.append(scraped_info)
            save_html_neet(app_no, html)

        elif status == "INVALID_AUTH":
            print(f"  ❌ Authentication failed for {app_no}.")
            failed_list.append({"Application No": app_no, "Reason": "INVALID_AUTH"})
        else:
            print(f"  ⚠️ Skipped after retries: {status}")
            failed_list.append({"Application No": app_no, "Reason": str(status)[:200]})

        # Polite delay between candidates to avoid triggering NIC rate limits
        time.sleep(2)

    # Final CSV summary
    if results_list:
        df = pd.DataFrame(results_list)
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"\n📊 Summary saved to {OUTPUT_FILE}")
        upload_to_drive(OUTPUT_FILE, "text/csv", "NEET_AdmitCards_2026")

    # NEW: Save failed list for retry later
    if failed_list:
        fail_df = pd.DataFrame(failed_list)
        fail_df.to_csv("NEET_FAILED.csv", index=False)
        print(f"⚠️ {len(failed_list)} failures logged to NEET_FAILED.csv")
        upload_to_drive("NEET_FAILED.csv", "text/csv", "NEET_AdmitCards_2026")

    print(f"\n🏁 ALL TASKS COMPLETED. Success: {len(results_list)} | Failed: {len(failed_list)}")

# --- TRIGGER ---
await main_neet_loop()


🚀 [1/53] Processing: 26041009998...
  🔑 Salt extracted: 3BW3237107455858
🔍 Solving captcha (pure ddddocr, max 15 attempts)...
  Attempt  1: Raw='Tcjxih' → Cleaned='TCJXIH' (len=6)
  ✅ VALID CAPTCHA: TCJXIH
  📍 POST URL: https://examinationservices.nic.in/AdmitCardService/AdmitCardNeet/Login  |  body: 10863 bytes  |  captcha: TCJXIH
  ❌ Authentication failed for 26041009998.

🚀 [2/53] Processing: 260410111171...
  🔑 Salt extracted: 96J6852A30669658
🔍 Solving captcha (pure ddddocr, max 15 attempts)...
  Attempt  1: Raw='SV240t' → Cleaned='SV240T' (len=6)
  ✅ VALID CAPTCHA: SV240T
  📍 POST URL: https://examinationservices.nic.in/AdmitCardService/AdmitCardNeet/Login  |  body: 10672 bytes  |  captcha: SV240T
  ❌ Authentication failed for 260410111171.

🚀 [3/53] Processing: 260410394044...
  🔑 Salt extracted: H3W7553088428677
🔍 Solving captcha (pure ddddocr, max 15 attempts)...
  Attempt  1: Raw='WFwEw' → Cleaned='WFWEW' (len=5)
  Attempt  2: Raw='0D2R9G' → Cleaned='0D2R9G' (len=6)
  ✅ VALI

In [ ]:
# --- STEP 10: FINAL CSV EXPORT & DRIVE UPLOAD ---

def save_csv_neet():
    """
    Finalizes the NEET extraction by uploading the central CSV
    summary to the Google Drive 'CSV_NEET_2026' subfolder.
    """
    # Using the output filename defined in Step 2
    file_path = "NEET_ADMIT_CARD.csv"

    if not os.path.exists(file_path):
        print("⚠️ No CSV found — it seems no successful results were fetched in this session.")
        return

    try:
        # 1. Encode the CSV file for transmission
        with open(file_path, "rb") as file:
            file_content = base64.b64encode(file.read()).decode("utf-8")

        # 2. Prepare the payload for the Apps Script Web App
        payload = {
            "fileContent": file_content,
            "filename": "NEET_Final_Summary_2026.csv",
            "mimeType": "text/csv",
            "subfolderName": "CSV_NEET_2026" # Distinct folder for CSV reports
        }

        print(f"📤 Uploading final summary to Drive...")

        # 3. Send the request
        response = requests.post(WEB_APP_URL, data=payload)

        # 4. Final Output check
        if "Success" in response.text:
            print(f"✅ Summary Upload Complete: {response.text}")
        else:
            print(f"⚠️ Drive Response: {response.text}")

    except Exception as e:
        print(f"❌ Critical error during CSV upload: {str(e)}")

# This should be called at the very end of your main_neet_loop()
save_csv_neet()